# Used-car QLoRA (Week 7, free Colab T4)

Fine-tune **Qwen2.5-1.5B-Instruct** to answer: *What does this used car cost to the nearest dollar?*

No OpenAI. No Modal. Use a T4 runtime.

Upload `train_sft.jsonl` from `car-deal-hunter/data/` (created by `python scripts/export_sft.py`) into the Colab Files pane, or point `JSONL` at a Drive path.

In [ ]:
!pip -q install -U transformers peft bitsandbytes accelerate datasets trl huggingface_hub

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
import torch

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
JSONL = "/content/train_sft.jsonl"
OUT = "/content/used-car-qlora"

assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
ds = load_dataset("json", data_files=JSONL, split="train")
ds = ds.shuffle(seed=42).select(range(min(4000, len(ds))))
print(ds[0]["text"][:400])

In [ ]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(
    model,
    LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    ),
)
model.print_trainable_parameters()

In [ ]:
args = SFTConfig(
    output_dir=OUT,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    report_to=[],
    optim="paged_adamw_8bit",
    max_length=256,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds,
    processing_class=tokenizer,
)
trainer.train()
trainer.model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
print("Download this folder into car-deal-hunter/data/qlora_adapter/")